In [1]:
import os
os.chdir("..")

In [2]:
%pwd

'e:\\MLOPS\\datascienceproject'

In [4]:
from dataclasses import dataclass
from pathlib import Path

@dataclass
class DataIngestionConfig:
    root_dir: Path
    source_URL: str
    local_data_file: Path
    unzip_dir: Path

In [15]:
from src.datascience.constants import *
from src.datascience.utils.common import read_yaml, create_directories
import urllib.request as request
from src.datascience import logger
import zipfile

In [11]:
class ConfigurationManager:
    def __init__(self, config_filepath=CONFIG_FILE_PATH, params_filepath=PARAMS_FILE_PATH, schema_filepath=SCHEMA_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)
        
        create_directories([self.config.artifacts_root])
        
    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion
        create_directories([config.root_dir])
        data_ingestion_config = DataIngestionConfig(
            root_dir=Path(config.root_dir),
            source_URL=config.source_URL,
            local_data_file=Path(config.local_data_file),
            unzip_dir=Path(config.unzip_dir)
        )
        
        return data_ingestion_config

         

In [19]:
class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config
        
    def download_data(self):
        if not self.config.local_data_file.exists():
            filename, headers = request.urlretrieve(self.config.source_URL, self.config.local_data_file)
            logger.info(f"Downloaded file: {filename}")
        else:
            logger.info(f"File already exists: {self.config.local_data_file}")
            
    def extract_zip_file(self):
        with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_ref:
            zip_ref.extractall(self.config.unzip_dir)
            logger.info(f"Extracted file to: {self.config.unzip_dir}")

In [20]:
try:
    config = ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion_config()
    data_ingestion = DataIngestion(config=data_ingestion_config)
    data_ingestion.download_data()
    data_ingestion.extract_zip_file()
except Exception as e:
    logger.exception(f"Error in data ingestion: {e}")

2026-02-25 02:15:52,216: INFO: common: yaml file: config\config.yaml loaded successfully
2026-02-25 02:15:52,228: INFO: common: yaml file: params.yaml loaded successfully
2026-02-25 02:15:52,230: INFO: common: yaml file: schema.yaml loaded successfully
2026-02-25 02:15:52,233: INFO: common: created directory at: artifacts
2026-02-25 02:15:52,235: INFO: common: created directory at: artifacts/data_ingestion
2026-02-25 02:15:52,237: INFO: 80134617: File already exists: artifacts\data_ingestion\data.zip
2026-02-25 02:15:52,264: INFO: 80134617: Extracted file to: artifacts\data_ingestion
